Run script
not debugged yet

In [ ]:
import os
import pickle
import numpy as np
import scipy.sparse as sp
from datetime import date
from multiprocessing import Pool

# Import user-defined functions (assumed available)
from blinking_utils import Determine_Blinking_Distribution5, DDC_MCMC

# Configuration
cluster = False  # Set True if using a cluster environment
Resolution = 40             # Determined via New_Determine_Res
stepper = 100               # Maximum number of MCMC steps
N_f = 100                   # Determined via Determine_N
Photon_weighted_Correction = True

# Load data: LocalizationsFinal, Frame_Information, filename
# (User should define how to load these, e.g., from .mat or .npz files)
# Example:
# data = np.load('data.npz', allow_pickle=True)
# LocalizationsFinal = data['LocalizationsFinal']
# Frame_Information = data['Frame_Information']
# filename = data['filename'].item()

# Placeholder for loaded data
LocalizationsFinal = []
Frame_Information = []
filename = 'experiment'
TrueLocalizations = []

# Initialize arrays
addonarray = np.zeros(500, dtype=int)
Photons = [[] for _ in Frame_Information]

# If Photons was not provided, initialize with ones
if not any(Photons):
    Photons = [np.ones(len(frames), dtype=float) for frames in Frame_Information]

# Initialize TrueLocalizations if empty
if not TrueLocalizations:
    TrueLocalizations = [[] for _ in Frame_Information]

# Ensure 3D localizations
for idx, loc in enumerate(LocalizationsFinal):
    loc = np.asarray(loc)
    if loc.shape[1] < 3:
        zcol = np.zeros((loc.shape[0], 1))
        LocalizationsFinal[idx] = np.hstack([loc, zcol])

# Prepare output containers
Final_Loc_Blinking_Corrected = [None] * len(Frame_Information)
Final_Frame_Blinking_Corrected = [None] * len(Frame_Information)
Trajectory_of_Localizations = [None] * len(Frame_Information)

# Determine blinking distribution
bins, Distribution_for_Blink, _, Resolution, X_overall, M_mat = \
    Determine_Blinking_Distribution5(
        LocalizationsFinal,
        Frame_Information,
        N_f,
        Resolution
    )

# Timer for each image (seconds)
timer = 3600 * 0.2

# Initialize MCMC tracking structures
n_images = len(Frame_Information)
Constantf = [None] * n_images
Constantf2 = [None] * n_images
Orderf = [None] * n_images
Step = np.zeros(n_images, dtype=int)
Lik = [None] * n_images
RelScore = [None] * n_images
Numb_of_Loc = [None] * n_images
Prob_dists = [dict(Deviation_in_Probability=None,
                   Prob_Distributions=None,
                   Dscale_store=None)
              for _ in range(n_images)]

# Prepare save path
today_str = date.today().isoformat()
output_name = f"Analyzed_Time_{today_str}_{filename}.pkl"

# Save initial state
with open(output_name, 'wb') as f:
    pickle.dump({
        'Constantf': Constantf,
        'Constantf2': Constantf2,
        'Orderf': Orderf,
        'Step': Step,
        'Lik': Lik,
        'RelScore': RelScore,
        'Numb_of_Loc': Numb_of_Loc,
        'Prob_dists': Prob_dists,
        'Final_Loc_Blinking_Corrected': Final_Loc_Blinking_Corrected,
        'Final_Frame_Blinking_Corrected': Final_Frame_Blinking_Corrected,
        'Trajectory_of_Localizations': Trajectory_of_Localizations,
    }, f)

# Blinking elimination loop
def process_image(args):
    ksu = args
    if len(LocalizationsFinal[ksu]) <= 10 or Step[ksu] >= stepper:
        Step[ksu] = stepper
        return None

    locs = LocalizationsFinal[ksu]
    frames = np.round(Frame_Information[ksu]).astype(int)
    if len(locs) > 8000:
        print(f"Warning: too many localizations in image {ksu}; consider splitting")

    result = DDC_MCMC(
        index=ksu,
        locs=locs,
        frames=frames,
        N_f=N_f,
        Resolution=Resolution,
        TrueLoc=TrueLocalizations[ksu],
        bins=bins,
        blink_dist=Distribution_for_Blink,
        step0=Step[ksu],
        timer=timer,
        Constf0=Constantf[ksu],
        Order0=Orderf[ksu],
        Lik0=Lik[ksu],
        RelScore0=RelScore[ksu],
        Numb0=Numb_of_Loc[ksu],
        Prob0=Prob_dists[ksu],
        stepper=stepper,
        addon=addonarray[ksu],
        Constf2_0=Constantf2[ksu],
        X_overall=X_overall,
        M_mat=M_mat
    )

    (Traj2,
     Final_loc2,
     Final_frame2,
     LikHood,
     Score,
     Numb,
     bestlik,
     steps,
     Constant,
     Order,
     Prob2,
     Constant2) = result

    maxtemp = np.max(Lik[ksu]) if Lik[ksu] is not None else -np.inf

    # Update probability distributions
    Prob_dists[ksu] = {
        'Deviation_in_Probability': sp.csr_matrix(Prob2['Deviation_in_Probability']),
        'Prob_Distributions': sp.csr_matrix(Prob2['Prob_Distributions']),
        'Dscale_store': Prob2['Dscale_store']
    }

    # Store MCMC state
    Constantf[ksu] = Constant
    Constantf2[ksu] = Constant2
    Orderf[ksu] = Order
    Step[ksu] = steps
    Lik[ksu] = LikHood
    RelScore[ksu] = Score
    Numb_of_Loc[ksu] = Numb

    if bestlik > maxtemp:
        if Photon_weighted_Correction:
            new_locs = []
            new_frames = []
            for traj_id in np.unique(Traj2):
                mask = (Traj2 == traj_id)
                weights = Photons[ksu][mask]
                weights = weights / weights.sum()
                wloc = np.average(locs[mask], axis=0, weights=weights)
                new_locs.append(wloc)
                new_frames.append(np.mean(Frame_Information[ksu][mask]))
            Final_loc2 = np.vstack(new_locs)
            Final_frame2 = np.array(new_frames)

        Final_Loc_Blinking_Corrected[ksu] = Final_loc2
        Final_Frame_Blinking_Corrected[ksu] = Final_frame2
        Trajectory_of_Localizations[ksu] = Traj2

    return None

# Main parallel or sequential loop
while np.any(Step < stepper):
    if cluster:
        # Example: use MPI or Slurm array jobs
        pass
    else:
        with Pool() as pool:
            pool.map(process_image, list(range(n_images)))

    # Save intermediate results
    with open(output_name, 'wb') as f:
        pickle.dump({
            'Constantf': Constantf,
            'Constantf2': Constantf2,
            'Orderf': Orderf,
            'Step': Step,
            'Lik': Lik,
            'RelScore': RelScore,
            'Numb_of_Loc': Numb_of_Loc,
            'Prob_dists': Prob_dists,
            'Final_Loc_Blinking_Corrected': Final_Loc_Blinking_Corrected,
            'Final_Frame_Blinking_Corrected': Final_Frame_Blinking_Corrected,
            'Trajectory_of_Localizations': Trajectory_of_Localizations,
        }, f)

# End of script


Determine_Blinking_Distribution5, debugged, working but with error percentage from least_square

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.optimize import least_squares, curve_fit, minimize
from scipy.io import loadmat
import warnings
import multiprocessing
from joblib import Parallel, delayed


def determine_blinking_distribution5(localizations_final, frame_information, pre_a, resolution):
    """
    This function determines P_{blink} from the supporting material
    and defines the bins used within the likelihood calculation.
    
    Parameters:
    -----------
    localizations_final : list of numpy arrays
        List of localization coordinates for each image
    frame_information : list of numpy arrays
        Frame information for each localization
    pre_a : int
        Threshold for blinking frame difference
    resolution : float
        Resolution for the bins
        
    Returns:
    --------
    bins : numpy array
        Bin edges used for histograms
    D_Counts3 : numpy array
        Probability distribution of blinking
    Total_No_Blink : list of numpy arrays
        No blinking distance distributions for each image
    Resolution : float
        Resolution value used
    X_overall : numpy array
        Overall blinking probability for each frame difference
    M_mat : numpy array
        Matrix M for the likelihood calculation
    """
    X_overall = []

    # First we are going to go through and find the max distance between locs in an image
    # We are also going to find the min distance in an image
    d_maxf = 0
    d_maxm = float('inf')
    for i in range(len(localizations_final)):
        print(f"{i+1}/{len(localizations_final)}")
        d = pdist(localizations_final[i])
        
        d_max = np.max(d)
        if d_max > d_maxf:
            d_maxf = d_max
        
        if d_max < d_maxm:
            d_maxm = d_max

    # Set the bins used in the likelihood calculation, this will be the same for every image
    # Key fix: Remove resolution from d_maxf in the range calculation
    bins = np.append(np.arange(0, d_maxf, resolution), np.inf)

    Total_Blink = []
    Total_No_Blink = []

    print('Working on step 1')

    for i in range(len(localizations_final)):
        # Here we will gather the distance distributions from as many cells as possible
        # This will allow us to define the joint probability distribution more accurately
        
        # Create a distance matrix for frame differences
        frame_zeros = np.zeros(len(frame_information[i]))
        frame_data = np.column_stack((frame_zeros, frame_information[i]))
        Z2 = pdist(frame_data)
        
        # Calculate pairwise distances between localizations
        D = pdist(localizations_final[i])
        
        # Separate distances into blinking and non-blinking categories
        D_Blink = D[Z2 < pre_a]
        
        D_No_Blink = D[(Z2 > pre_a) & (Z2 < pre_a * 5)]
        
        # Calculate the probability distributions (MATLAB-like approach)
        counts, _ = np.histogram(D_Blink, bins=bins, density=False)
        D_Counts = counts / counts.sum() if counts.sum() > 0 else counts
        Total_Blink.append(D_Counts)
        
        counts2, _ = np.histogram(D_No_Blink, bins=bins, density=False)
        D_Counts2 = counts2 / counts2.sum() if counts2.sum() > 0 else counts2
        Total_No_Blink.append(D_Counts2)

    # Convert to numpy arrays for easier manipulation
    Total_Blink = np.array(Total_Blink)
    Total_No_Blink = np.array(Total_No_Blink)

    # Here we go through and scale the probability distributions after 10 times
    # the resolution of the experiment
    D_Counts = np.mean(Total_Blink, axis=0)
    D_Counts2 = np.mean(Total_No_Blink, axis=0)
    
    # Save a copy of True_Distribution for CSV export later
    True_Distribution = D_Counts2.copy()

    # Fix for consistent D_Counts3 calculation:
    # Scale the non-blinking distribution to match the blinking distribution at larger distances
    # Use index 9: instead of 10: to match manual cell calculation
    D_Scale = np.sum(D_Counts[9:]) / np.sum(D_Counts2[9:])
    D_Counts3 = D_Counts - D_Counts2 * D_Scale
    D_Counts3 = D_Counts3 / np.sum(D_Counts3)

    # Clean up the distributions to be consistent with how the distribution should behave
    good = True
    ins = 0
    for i in range(3, len(D_Counts3) - 1):  # Start from index 3 to match manual calculation
        if D_Counts3[i] > 0 and D_Counts3[i+1] < D_Counts3[i] and good:
            continue
        else:
            if ins == 0:
                ins = i
            good = False
            D_Counts3[i] = 0
    
    D_Counts3[D_Counts3 < 0] = 0
    D_Counts3 = D_Counts3 / np.sum(D_Counts3)
    D_Counts3 = D_Counts3 / np.sum(D_Counts3)  # Normalize again to ensure sum=1

    # Fix for Noise Elimination: Use index 7: instead of 8: to match manual calculation
    # Anything greater than 8 times the resolution is considered noise
    if np.sum(D_Counts3[7:] > 0) > 1:
        print('Warning: Eliminating Noise for higher bins')
        D_Counts3[7:] = 0
        D_Counts3 = D_Counts3 / np.sum(D_Counts3)

    Distribution_for_Blink2 = D_Counts3

    # Here we do the fitting to determine how much of each distribution makes up
    # the pairwise distance distributions at each frame difference
    # This is equation 3 of the Supporting Material
    Dscale_store = [[] for _ in range(len(localizations_final))]
    
    print('Still Working on step 1, wait for me')

    # Define function for parallel processing
    def process_image(i):
        print(f'Processing image {i+1}/{len(localizations_final)}')
        
        # Create distance matrix for frame differences
        frame_zeros = np.zeros(len(frame_information[i]))
        frame_data = np.column_stack((frame_zeros, frame_information[i]))
        Z2 = pdist(frame_data)
        
        # Calculate pairwise distances between localizations
        D = pdist(localizations_final[i])
        
        # Get non-blinking distances
        D_No_Blink = D[(Z2 > pre_a) & (Z2 < pre_a * 5)]
        counts, _ = np.histogram(D_No_Blink, bins=bins, density=False)
        True_Distribution2 = counts / counts.sum() if counts.sum() > 0 else counts
        
        dscale_results = []

        def fit_scale_trf(T, B, y):
            """Fit y ≈ x*T + (1-x)*B with TRF (SciPy version)."""
            def residual(x):
                return (x[0]*T + (1-x[0])*B) - y

            res = least_squares(
            residual,
            x0=[1.0],
            bounds=(0.0, 1.0),
            method="trf",       # Trust Region Reflective
            ftol=1e-6,
            xtol=1e-6,
            gtol=1e-6
            )
            return res.x[0]
        
        for w in range(1, pre_a + 1):
            D_Blink = D[Z2 == w]
            counts, _ = np.histogram(D_Blink, bins=bins, density=False)
            Temp_Distribution = counts / counts.sum() if counts.sum() > 0 else counts
            
            y = Temp_Distribution
            # t = np.vstack((True_Distribution2, Distribution_for_Blink2))
            
            # # Define objective function for curve fitting
            # def F(x):
            #     return x * t[0, :] + (1 - x) * t[1, :]
            
            # # Simple optimization with constraints
            # x0 = 1.0
            # bounds = (0, 1)
            
            # # Custom fitting with bounded values
            # def residuals(x):
            #     return y - F(x)
                
            # with warnings.catch_warnings():
            #     warnings.simplefilter("ignore")
            #     result = minimize(lambda x: np.sum(residuals(x)**2), x0=x0, bounds=[bounds])
            #     x = result.x[0]

            D_Scale = fit_scale_trf(True_Distribution2, Distribution_for_Blink2, y)
            dscale_results.append(D_Scale)
        
        return dscale_results

    # Use parallel processing when available
    try:
        num_cores = multiprocessing.cpu_count()
        results = Parallel(n_jobs=min(num_cores, len(localizations_final)))(
            delayed(process_image)(i) for i in range(len(localizations_final))
        )
        
        for i, result in enumerate(results):
            Dscale_store[i] = result
    except:
        print("Parallel processing failed, falling back to serial processing")
        for i in range(len(localizations_final)):
            Dscale_store[i] = process_image(i)

    # Convert list of lists to 2D array
    Dscale_store2 = np.array(Dscale_store)

    if len(localizations_final) > 1:
        X_overall = np.mean(Dscale_store2, axis=0)
        Dscale_store = np.mean(Dscale_store2, axis=0)
    else:
        X_overall = Dscale_store2[0]
        Dscale_store = Dscale_store2[0]

    # Matrix M calculation
    # Better to do it over all images for less error with lower number of localizations
    Dscale_store = np.array(Dscale_store)
    Dscale_store[Dscale_store > 1] = 1
    Dscale_store[Dscale_store < 0] = 0.0000001
    if len(Dscale_store) > 0:  # Ensure there's something to index
        Dscale_store[-1] = 1
    
    Deviation_in_Probabilityt = [[] for _ in range(len(localizations_final))]
    
    print('Still going, Working on step 1')
    
    # Define function for parallel processing of M matrix calculation
    def process_image_m_matrix(i):
        print(f'Processing M matrix for image {i+1}/{len(localizations_final)}')
        
        # Create distance matrix for frame differences
        frame_zeros = np.zeros(len(frame_information[i]))
        frame_data = np.column_stack((frame_zeros, frame_information[i]))
        Z2 = pdist(frame_data)
        
        # Calculate pairwise distances between localizations
        D = pdist(localizations_final[i])
        
        D_No_Blink = D[(Z2 > pre_a) & (Z2 < pre_a * 5)]
        counts, _ = np.histogram(D_No_Blink, bins=bins, density=False)
        True_Distribution = counts / counts.sum() if counts.sum() > 0 else counts
        
        deviation_results = []
        for w in range(1, pre_a + 1):
            D_Scale = Dscale_store[w-1]
            
            Temp_Distribution2 = Distribution_for_Blink2 * (1 - D_Scale) + True_Distribution * D_Scale
            
            # See equation in determining probability section of paper
            with np.errstate(divide='ignore', invalid='ignore'):
                Combined = (Temp_Distribution2 - D_Scale * True_Distribution) / Temp_Distribution2
                # Handle division by zero
                Combined[~np.isfinite(Combined)] = 0
            
            deviation_results.append(Combined)
        
        return deviation_results

    # Use parallel processing when available
    try:
        num_cores = multiprocessing.cpu_count()
        results = Parallel(n_jobs=min(num_cores, len(localizations_final)))(
            delayed(process_image_m_matrix)(i) for i in range(len(localizations_final))
        )
        
        for i, result in enumerate(results):
            Deviation_in_Probabilityt[i] = result
    except:
        print("Parallel processing failed, falling back to serial processing")
        for i in range(len(localizations_final)):
            Deviation_in_Probabilityt[i] = process_image_m_matrix(i)

    # Calculate M matrix
    if len(localizations_final) > 1:
        M_mat = np.zeros((pre_a, len(bins)-1))
        for w in range(pre_a):
            M_mat_t = np.array([Deviation_in_Probabilityt[i][w] for i in range(len(localizations_final))])
            M_mat[w, :] = np.mean(M_mat_t, axis=0)
    else:
        M_mat = np.array(Deviation_in_Probabilityt[0])

    # Optional: calculate median version (we don't use this, but just in case)
    M_mat2 = np.zeros((pre_a, len(bins)-1))
    for w in range(pre_a):
        M_mat_t = np.array([Deviation_in_Probabilityt[i][w] for i in range(len(localizations_final))])
        M_mat2[w, :] = np.median(M_mat_t, axis=0)

    # Save intermediate results to CSV for debugging or analysis
    # np.savetxt('Dscale_store_python.csv', Dscale_store, delimiter=',')
    # np.savetxt('X_overall_python.csv', X_overall, delimiter=',')
    # np.savetxt('M_mat_python.csv', M_mat, delimiter=',')
    # np.savetxt('True_Distribuiton_py.csv', True_Distribution, delimiter=',')
    
    # deviation_example = np.array(Deviation_in_Probabilityt[0])
    # np.savetxt('Deviation_in_Probabilityt_py.csv', deviation_example.reshape(-1), delimiter=',')
    # np.savetxt('Deviation_in_Probabilityt_index_py.csv', np.arange(len(deviation_example.reshape(-1))), delimiter=',')

    return bins, D_Counts3, Total_No_Blink, resolution, X_overall, M_mat